## CA 4 - Part 2, LLMs Spring 2025

- **Name: Mohammad Amanlou**
- **Student ID: 810100084**

---
#### Your submission should be named using the following format: `CA4_LASTNAME_STUDENTID.ipynb`.

---

TA Email: miladmohammadi@ut.ac.ir

##### *How to do this problem set:*

- Some questions require writing Python code and computing results, and the rest of them have written answers. For coding problems, you will have to fill out all code blocks that say `YOUR CODE HERE`.

- For text-based answers, you should replace the text that says ```Your Answer Here``` with your actual answer.

- There is no penalty for using AI assistance on this homework as long as you fully disclose it in the final cell of this notebook (this includes storing any prompts that you feed to large language models). That said, anyone caught using AI assistance without proper disclosure will receive a zero on the assignment (we have several automatic tools to detect such cases). We're literally allowing you to use it with no limitations, so there is no reason to lie!

---

##### *Academic honesty*

- We will audit the Colab notebooks from a set number of students, chosen at random. The audits will check that the code you wrote actually generates the answers in your notebook. If you turn in correct answers on your notebook without code that actually generates those answers, we will consider this a serious case of cheating.

- We will also run automatic checks of Colab notebooks for plagiarism. Copying code from others is also considered a serious case of cheating.

---

## Text2SQL

In this section, you will progressively build and evaluate multiple Text-to-SQL pipelines. You’ll start with a simple prompting-based baseline, then design a graph-based routing system using chain-of-thought and schema reasoning, and finally construct a ReAct agent that interacts with the schema via tools. Each stage demonstrates a different strategy for generating SQL from natural language using LLMs.

### Initializations

This section prepares the environment and initializes the LLM model (Gemini) to be used in later parts of the notebook.

In [1]:
%pip install -r requirements.txt

  Obtaining dependency information for langgraph from https://files.pythonhosted.org/packages/17/8a/fe05ec63ee4c3889a8b89679a6bdd1be6087962818996f3b361da23a5529/langgraph-0.4.8-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-google-genai from https://files.pythonhosted.org/packages/5e/70/0747358eca996f713f715e2bfc2d0805804f8f705af57381fbee91bb475a/langchain_google_genai-2.1.5-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/44.3 kB ? eta -:--:--
     ---------------------------------------- 44.3/44.3 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Obtaining dependency information for langgraph-checkpoint>=2.0.26 from https://files.pythonhosted.org/packages/38/48/d7cec540a3011b3207470bb07294a399e3b94b2e8a602e38cb007ce5bc10/langgraph_checkpoint-2.0.26-py3-none-any.whl.metadata
  Obtaining dependency information for langgraph-prebuilt>=0.2.0 from https://fi

#### Load API Key (2 Points)

**Task:** Load the Gemini API key stored in the `.env` file and set it as an environment variable so it can be used to authenticate API requests later.

* Use `dotenv` to load the file.
* Extract the API key with `os.getenv`.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
gemini_api_key = os.getenv("G_API_KEY")

if gemini_api_key is None:
    raise RuntimeError("GEMINI_API_KEY is not set in your environment variables")


#### Create ChatModel (3 Points)

**Task:** Create an instance of the Gemini LLM using LangChain. You should configure the model with proper parameters for our task.

Note: You may use any model that supports Structured Output and Tool Use. We recommend using gemini-2.5-flash-preview-05-20 from Google AI Studio, as it offers a generous free tier.

In [2]:
import os
from dotenv import load_dotenv
from langchain.tools import Tool
from langchain.agents import initialize_agent, AgentType
from langchain_google_genai import ChatGoogleGenerativeAI
from func_timeout import func_timeout, FunctionTimedOut
from tqdm import tqdm
import pandas as pd

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL_NAME = "gemini-2.5-flash-preview-05-20"

gemini_chat = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=0, api_key=gemini_api_key)


### Baseline

In this section, you'll build a simple baseline pipeline that directly converts a question and schema into a SQL query using a single prompt.

#### Baseline Function (5 Points)

**Task:** Implement a function that sends a system message defining the task, and a user message containing the input question and schema. The LLM should return the SQL query formatted as: "```sql\n[query]```"

In [4]:
from langchain.schema import SystemMessage, HumanMessage

def run_baseline(question: str, schema: str) -> str:
    # 1) Define the system message to tell the model exactly what to do
    system_msg = SystemMessage(
        content=(
            "You are a SQL generation assistant. "
            "When given a user question and a database schema, "
            "you must output a single SQL query that answers the question. "
            "Format your output *exactly* as:\n"
            "```sql\n"
            "[your SQL here]\n"
            "```"
        )
    )

    # 2) Pack the user content with the question and the schema
    user_msg = HumanMessage(
        content=(
            f"Question:\n{question}\n\n"
            f"Schema:\n{schema}"
        )
    )

    # 3) Call the chat model
    response = gemini_chat(messages=[system_msg, user_msg])

    # 4) Return the assistant’s reply
    return response.content


#### Run and Evaluate (Estimated Run Time 5-10min)

Run your baseline function over the dataset provided.

In [5]:
from method_run import run_method
import re

def function_template(item):
    result = run_baseline(item['question'], item['schema'])
    # First try to extract query from markdown SQL block
    match = re.search(r'```sql\n(.*?)```', result, re.DOTALL)
    if match:
        query = match.group(1).strip()
    else:
        # If no markdown block found, try to extract just SQL query
        query = result.strip()
        # Remove any ```sql or ``` if present without proper formatting
        query = re.sub(r'```sql|```', '', query).strip()
    
    print(f"Question: {item['question']}")
    print(f"Schema: {item['schema']}")
    print(f"Generated SQL: {query}\n")
    
    return {**item, 'sql': query}

run_method(function_template, SLEEP_TIME=10)

#Run on mode=nano if you want to test it on a smaller dataset
#run_method(function_template, SLEEP_TIME=10, mode="nano")

  0%|          | 0/18 [00:00<?, ?it/s]C:\Users\a\AppData\Local\Temp\ipykernel_21732\3089945793.py:26: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use invoke instead.
  response = gemini_chat(messages=[system_msg, user_msg])


Question: Find the percentage of atoms with single bond. (Evidence: single bond refers to bond_type = '-'; percentage = DIVIDE(SUM(bond_type = '-'), COUNT(bond_id)) as percentage)
Schema: atom (atom_id, molecule_id, element)
bond (bond_id, molecule_id, bond_type)
connected (atom_id, atom_id2, bond_id)
molecule (molecule_id, label)

Generated SQL: SELECT
  CAST(SUM(CASE WHEN bond_type = '-' THEN 1 ELSE 0 END) AS REAL) * 100 / COUNT(bond_id)
FROM bond;



  6%|▌         | 1/18 [00:12<03:29, 12.33s/it]

Question: Indicate which atoms are connected in non-carcinogenic type molecules. (Evidence: label = '-' means molecules are non-carcinogenic)
Schema: atom (atom_id, molecule_id, element)
bond (bond_id, molecule_id, bond_type)
connected (atom_id, atom_id2, bond_id)
molecule (molecule_id, label)

Generated SQL: SELECT
  T1.atom_id,
  T1.atom_id2
FROM connected AS T1
INNER JOIN bond AS T2
  ON T1.bond_id = T2.bond_id
INNER JOIN molecule AS T3
  ON T2.molecule_id = T3.molecule_id
WHERE
  T3.label = '-';



 11%|█         | 2/18 [00:32<04:31, 16.96s/it]

...

Question: Calculate the total average cost that Elijah Allen spent in the events on September and October. (Evidence: events in September and October refers to month(expense_date) = 9 AND MONTH(expense_date) = 10)
Schema: event (event_id, event_name, event_date, type, notes, location, status)
major (major_id, major_name, department, college)
zip_code (zip_code, type, city, county, state, short_state)
attendance (link_to_event, link_to_member)
budget (budget_id, category, spent, remaining, amount, event_status, link_to_event)
expense (expense_id, expense_description, expense_date, cost, approved, link_to_member, link_to_budget)
income (income_id, date_received, amount, source, notes, link_to_member)
member (member_id, first_name, last_name, email, position, t_shirt_size, phone, zip, link_to_major)

Generated SQL: SELECT
  AVG(T1.cost)
FROM expense AS T1
INNER JOIN member AS T2
  ON T1.link_to_member = T2.member_id
WHERE
  T2.first_name = 'Elijah' AND T2.last_name = 'Allen' AND (
    STR

 94%|█████████▍| 17/18 [03:38<00:11, 11.77s/it]

Question: Find the name and date of events with expenses for pizza that were more than fifty dollars but less than a hundred dollars. (Evidence: name of event refers to event_name; date of event refers to event_date; expenses for pizza refers to expense_description = 'Pizza' where cost > 50 and cost < 100)
Schema: event (event_id, event_name, event_date, type, notes, location, status)
major (major_id, major_name, department, college)
zip_code (zip_code, type, city, county, state, short_state)
attendance (link_to_event, link_to_member)
budget (budget_id, category, spent, remaining, amount, event_status, link_to_event)
expense (expense_id, expense_description, expense_date, cost, approved, link_to_member, link_to_budget)
income (income_id, date_received, amount, source, notes, link_to_member)
member (member_id, first_name, last_name, email, position, t_shirt_size, phone, zip, link_to_major)

Generated SQL: SELECT
  T1.event_name,
  T1.event_date
FROM event AS T1
INNER JOIN budget AS T2
 

100%|██████████| 18/18 [03:49<00:00, 12.76s/it]


Starting to compare without knowledge for ex
Process finished successfully
start calculate
                     simple               moderate             challenging          total               
count                6                    6                    6                    18                  
======================================    ACCURACY    =====================================
accuracy             66.67                83.33                83.33                77.78               
Finished evaluation



### Chain/Router

Here, you will build a more advanced system that routes the query through different paths based on question difficulty. Easier questions go straight to query generation; harder ones go through schema path extraction first.

#### Define State (5 Points)

**Task:** Define a `RouterGraphState` using `MessagesState` and `pydantic` that contains:
* The input question and schema
* The predicted difficulty level
* The extracted schema path
* The final query

In [35]:

from typing import Literal, Optional
from pydantic import BaseModel, Field
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain.schema import SystemMessage, HumanMessage
import re

In [80]:
class RouterGraphState(MessagesState):
    question: str = Field(..., description="The user's natural question")
    schema: str = Field(..., description="The database schema")
    question_difficulty: Literal["easy", "moderate", "challenging"] = Field(
        ..., description="Predicted difficulty"
    )
    schema_path: Optional[str] = Field(
        None, description="Join path extracted for hard questions"
    )
    query: Optional[str] = Field(
        None, description="The generated SQL query"
    )

#### Node: Analyser (5 Points)

**Task:** Build a node that:
* Accepts a question and schema
* Analyzes the difficulty (simple/moderate/challanging)
* Uses the LLM’s structured output feature to return the difficulty

**Steps**:

1. Define a Pydantic class to hold the expected structured output.
2. Use structure output mode of LLM to bind it to the model.

In [61]:
from pydantic import BaseModel
import json

class QuestionDifficultyAnalysis(BaseModel):
    question_difficulty: Literal["easy", "moderate", "challenging"]

def analyser_node(state):
    # ensure dict form
    if not isinstance(state, dict):
        state = state.dict()

    system = SystemMessage(
        content=(
            "You are a difficulty classifier.  "
            "Classify the question into exactly one of: easy, moderate, challenging.  "
            "Respond *only* with a JSON object like:\n"
            '{ "question_difficulty": "easy" }'
        )
    )
    user = HumanMessage(content=f"Question:\n{state['question']}\n\nSchema:\n{state['schema']}")

    # 1) call the LLM
    resp = gemini_chat(messages=[system, user])
    text = resp.content.strip()

    # 2) parse JSON
    try:
        data = json.loads(text)
        state["question_difficulty"] = data["question_difficulty"]
    except Exception:
        # fallback: if it printed extra, try to extract the JSON block
        import re
        m = re.search(r'\{.*?\}', text, re.DOTALL)
        data = json.loads(m.group(0)) if m else {}
        state["question_difficulty"] = data.get("question_difficulty", "easy")

    return state

#### Conditional Edge (2 Points)

**Task:** Implement a branching function that decides whether to proceed to direct query generation or schema path extraction based on the difficulty label returned by the analyser.

* If the difficulty is “easy”, go directly to query generation.
* Otherwise, extract the schema path first.

In [62]:
def is_schema_extraction_needed(state: RouterGraphState) -> Literal["schema_path_extractor", "query_generator"]:
    return "query_generator" if state["question_difficulty"] == "easy" else "schema_path_extractor"


#### Node: Schema Extractor (3 Points)

**Task:** Implement a node that takes the question and schema and extracts a join path or sequence of relevant tables from the schema based on the question.

* Use a simple prompt for this.
* Store the result in the `schema_path` field of the state.

In [63]:
def schema_path_extractor_node(state):
    if not isinstance(state, dict):
        state = state.dict()

    question = state["question"]
    schema   = state["schema"]

    system = SystemMessage(
        content=(
            "You are a schema-path extractor.  "
            "Given a question and a schema, output the minimal join path "
            "(tables and join keys) needed to answer it, as plain text."
        )
    )
    user = HumanMessage(content=f"Question:\n{question}\n\nSchema:\n{schema}")
    resp = gemini_chat(messages=[system, user])

    state["schema_path"] = resp.content.strip()
    return state

#### Node: Generator (5 Points)

**Task:** Generate the SQL query based on the question and schema.

* If a schema path is available, include it in the prompt.
* Save the output query in the `query` field of the state.


In [64]:
import re
from langchain.schema import SystemMessage, HumanMessage

def query_generator_node(state):
    if not isinstance(state, dict):
        state = state.dict()

    question = state["question"]
    schema   = state["schema"]
    path     = state.get("schema_path")

    path_instr = f"Use this join path:\n{path}\n\n" if path else ""
    system = SystemMessage(
        content=(
            "You are a SQL assistant.  "
            "Produce exactly one SQL query wrapped in a markdown block:\n"
            "```sql\n<your query>\n```"
        )
    )
    user = HumanMessage(content=f"{path_instr}Question:\n{question}\n\nSchema:\n{schema}")
    raw = gemini_chat(messages=[system, user]).content

    m = re.search(r"```sql\s*(.*?)```", raw, re.DOTALL)
    sql = m.group(1).strip() if m else raw.strip().strip("```").strip()

    state["query"] = sql
    return state

#### Build Graph (5 Points)

**Task:** Assemble the full routing graph using the nodes and edges you created.

In [65]:
builder = StateGraph(RouterGraphState)

builder.add_node("analyser", analyser_node)
builder.add_node("schema_path_extractor", schema_path_extractor_node)
builder.add_node("query_generator", query_generator_node)

builder.add_edge(START, "analyser")
builder.add_conditional_edges("analyser", is_schema_extraction_needed)
builder.add_edge("schema_path_extractor", "query_generator")
builder.add_edge("query_generator", END)

router_graph = builder.compile()

#### Run and Evaluate (Estimated Run Time 10-15min)

**Task:** Run your compiled routing graph on a dataset. For each question:

* Instantiate the `RouterGraphState` with the question and schema.
* Run the graph to completion.
* Extract and clean the query from the result.

Use the `run_method` function to handle iteration and timing.

In [47]:

from method_run import run_method
def run_router_graph(item):
    response = router_graph.invoke(
        RouterGraphState(
            question=item['question'],
            schema=item['schema'],
            schema_path=None,
            question_difficulty=None,
            query=None
        )
    )
    result = response["query"]
    # First try to extract query from markdown SQL block
    match = re.search(r'```sql\n(.*?)```', result, re.DOTALL)
    if match:
        query = match.group(1).strip()
    else:
        # If no markdown block found, try to extract just SQL query
        query = result.strip()
        # Remove any ```sql or ``` if present without proper formatting
        query = re.sub(r'```sql|```', '', query).strip()
    print(f"Question: {item['question']}")
    print(f"Schema: {item['schema']}")
    print(f"Question Difficulty: {response['question_difficulty']}")
    if response["schema_path"]:
        print(f"Schema Path: {response['schema_path']}")
    print(f"Generated SQL: {query}\n")
    return {**item, 'sql': query}


run_method(run_router_graph, SLEEP_TIME=30)

#Run on mode=nano if you want to test it on a smaller dataset
#run_method(run_router_graph, SLEEP_TIME=10, mode="nano")

  0%|          | 0/18 [00:00<?, ?it/s]

Question: Find the percentage of atoms with single bond. (Evidence: single bond refers to bond_type = '-'; percentage = DIVIDE(SUM(bond_type = '-'), COUNT(bond_id)) as percentage)
Schema: atom (atom_id, molecule_id, element)
bond (bond_id, molecule_id, bond_type)
connected (atom_id, atom_id2, bond_id)
molecule (molecule_id, label)

Question Difficulty: moderate
Schema Path: bond
Generated SQL: SELECT
  SUM(CASE WHEN bond_type = '-' THEN 1 ELSE 0 END) * 100.0 / COUNT(bond_id)
FROM bond;



  6%|▌         | 1/18 [00:42<11:58, 42.24s/it]

Question: Indicate which atoms are connected in non-carcinogenic type molecules. (Evidence: label = '-' means molecules are non-carcinogenic)
Schema: atom (atom_id, molecule_id, element)
bond (bond_id, molecule_id, bond_type)
connected (atom_id, atom_id2, bond_id)
molecule (molecule_id, label)

Question Difficulty: moderate
Schema Path: connected (bond_id)
bond (bond_id, molecule_id)
molecule (molecule_id, label)
Generated SQL: SELECT
  c.atom_id,
  c.atom_id2
FROM connected AS c
JOIN bond AS b
  ON c.bond_id = b.bond_id
JOIN molecule AS m
  ON b.molecule_id = m.molecule_id
WHERE
  m.label = '-';



 11%|█         | 2/18 [01:28<11:55, 44.70s/it]

...

 94%|█████████▍| 17/18 [10:49<00:39, 39.06s/it]

Question: Find the name and date of events with expenses for pizza that were more than fifty dollars but less than a hundred dollars. (Evidence: name of event refers to event_name; date of event refers to event_date; expenses for pizza refers to expense_description = 'Pizza' where cost > 50 and cost < 100)
Schema: event (event_id, event_name, event_date, type, notes, location, status)
major (major_id, major_name, department, college)
zip_code (zip_code, type, city, county, state, short_state)
attendance (link_to_event, link_to_member)
budget (budget_id, category, spent, remaining, amount, event_status, link_to_event)
expense (expense_id, expense_description, expense_date, cost, approved, link_to_member, link_to_budget)
income (income_id, date_received, amount, source, notes, link_to_member)
member (member_id, first_name, last_name, email, position, t_shirt_size, phone, zip, link_to_major)

Question Difficulty: moderate
Schema Path: event.event_id = budget.link_to_event
budget.budget_id

100%|██████████| 18/18 [11:27<00:00, 38.18s/it]

Starting to compare without knowledge for ex


Process finished successfully
start calculate
                     simple               moderate             challenging          total               
count                6                    6                    6                    18                  
======================================    ACCURACY    =====================================
accuracy             50.00                83.33                83.33                72.22               
Finished evaluation



### Agent (ReAct)

Now you will implement a full ReAct agent that incrementally solves the Text-to-SQL task using tools. The agent can explore tables and columns before finalizing the query.

**You are not allowed to use 'Prebuilt Agent' of LangGraph. You have to build your own graph.**

#### Define Tools

**Task:** Define three tools for the agent to interact with the schema:
1. `get_samples_from_table`: Returns the first few rows of a table.
2. `get_column_description`: Provides a human-readable description of a specific column.
3. `execute`: Executes a SQL query.

In [66]:
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig
from langgraph.prebuilt import ToolNode

from db_manager import DBManager
db_manager = DBManager()

@tool
def get_samples_from_table(table_name: str, config: RunnableConfig):
  """Gets the first few rows (samples) from a specified table.

  Args:
    table_name: The name of the table from which to fetch samples.

  Returns:
    The first few rows from the specified table.
  """
  db_name = config["configurable"].get("database_name")
  result = db_manager.get_table_head(table_name, db_name=db_name)
  return result

@tool
def get_column_description(table_name: str, column_name: str, config: RunnableConfig):
  """Provides a description for a specific column within a given table.

  Args:
    table_name: The name of the table containing the column.
    column_name: The name of the column for which to get the description.

  Returns:
    A string containing the description of the specified column.
  """
  db_name = config["configurable"].get("database_name")
  result = db_manager.get_column_description(db_name, table_name, column_name)
  return result

@tool
def execute(query: str, config: RunnableConfig):
  """Executes a given SQL query against the database.

  Args:
    query: The SQL query string to be executed.

  Returns:
    The result of the executed query. This could be a set of rows,
    a confirmation message, or an error.
  """
  db_name = config["configurable"].get("database_name")
  result = db_manager.query(query, db_name)
  return result

#### Extra Tool (5+5 Bonus Points):

**Task**: Create and integrate a new custom tool into the ReAct agent. To receive credit for this part, your tool must be meaningfully different from the existing three tools and provide practical value in helping the agent generate more accurate or efficient SQL queries.

In [67]:
@tool
def get_distinct_column_values(
    table_name: str,
    column_name: str,
    config: RunnableConfig,
    limit: int = 10
):
    """Returns up to `limit` distinct values from the specified column.
    
    This helps the agent understand the domain of a column when constructing
    WHERE clauses or filters.
    """
    db_name = config["configurable"].get("database_name")
    query = f"""
        SELECT DISTINCT {column_name}
        FROM {table_name}
        LIMIT {limit};
    """
    return db_manager.query(query, db_name=db_name)

@tool
def get_foreign_keys(table_name: str, config: RunnableConfig):
    """Lists all foreign-key relationships for the given table.

    Returns a list of (constraint_name, source_column, target_table, target_column)
    so the agent can discover how tables join together.
    """
    db_name = config["configurable"].get("database_name")
    sql = f"""
    SELECT
      tc.constraint_name,
      kcu.column_name AS source_column,
      ccu.table_name  AS target_table,
      ccu.column_name AS target_column
    FROM information_schema.table_constraints AS tc
    JOIN information_schema.key_column_usage AS kcu
      ON tc.constraint_name = kcu.constraint_name
     AND tc.table_name = kcu.table_name
    JOIN information_schema.constraint_column_usage AS ccu
      ON ccu.constraint_name = tc.constraint_name
    WHERE tc.constraint_type = 'FOREIGN KEY'
      AND tc.table_name = '{table_name}';
    """
    return db_manager.query(sql, db_name=db_name)


@tool
def get_table_row_count(table_name: str, config: RunnableConfig):
    """Returns the exact number of rows in the specified table.

    This helps the agent estimate the table’s size and choose appropriate filters or joins.
    """
    db_name = config["configurable"].get("database_name")
    sql = f"SELECT COUNT(*) AS row_count FROM {table_name};"
    result = db_manager.query(sql, db_name=db_name)
    # Expect result like [{'row_count': 12345}]
    if isinstance(result, list) and result:
        return result[0].get("row_count", result)
    return result


#### Create Tool Node

In [68]:
tools = [get_samples_from_table, get_column_description, execute, get_distinct_column_values,get_foreign_keys, get_table_row_count ]
tools_node = ToolNode(tools=tools)

#### ReAct Agent Prompt (5 Points)

**Task:** Set up the agent node with planning, tool use, and final SQL generation prompts. For writing efficient prompt you can read this link.
https://cookbook.openai.com/examples/gpt4-1_prompting_guide

In [84]:
REACT_SYS_PROMPT = """
You are an expert AI assistant that generates correct, efficient SQL queries from natural‐language questions.
You have access to the following tools:

• get_samples_from_table(table_name)  
• get_column_description(table_name, column_name)  
• get_distinct_column_values(table_name, column_name, limit)  
• get_foreign_keys(table_name)  
• get_table_row_count(table_name)  
• execute(query)  

When you get a question and its schema, follow the ReAct pattern exactly:

Thought: <think about what you need to know or do next>  
Action: <one of the tool names>  
Action Input: <the input for that tool>  
Observation: <the output of the tool>  

…repeat Thought/Action/Action Input/Observation until you have enough information…

Finally, produce:

Final Answer:  
```sql
<your complete SQL query here>
— Only call a tool when you need more information about the schema or data.
— Do not output any extra text outside the Thought/Action/Observation/Final Answer structure.
— Use the schema and any observations to ensure your SQL uses the correct table and column names.
"""



#### Agent Node (5 Points)

**Task:** Set up the agent node with models that have binded with tools.

In [88]:
import time
agent_llm = gemini_chat.bind_tools(tools)
def agent_node(state: MessagesState) -> MessagesState:
    # For rate‐limiting purposes, sleep before we call the LLM
    time.sleep(10)

    # 1) Build the ReAct system prompt
    sys_msg = SystemMessage(content=REACT_SYS_PROMPT)

    # 2) Grab the existing message history from the state
    history = state["messages"]  # assuming MessagesState.messages is your list of prior messages

    # 3) Invoke the tool‐enabled LLM with the prompt + history
    #    This returns an updated list of messages including thoughts, tool calls, and final answer
    new_history = agent_llm.invoke([sys_msg] + history)

    # 4) Write the new history back into the state and return
    state["messages"] = new_history
    return state

Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'default' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring


#### Build Graph (5 Points)

**Task:** Assemble the ReAct agent graph, connecting the agent node and tool node.

In [89]:
from langgraph.prebuilt import tools_condition
from typing_extensions import TypedDict

class ConfigSchema(TypedDict):
    database_name: str

react_builder = StateGraph(MessagesState, config_schema=ConfigSchema)

from langgraph.graph import START

# 1) Register the agent node and the tools node
react_builder.add_node("agent", agent_node)
react_builder.add_node("tools", tools_node)

# 2) Start by running the agent
react_builder.add_edge(START, "agent")

# 3) After the agent’s turn, branch: if it chose a tool, run the tools node
react_builder.add_conditional_edges("agent", tools_condition)

# 4) Once a tool has run, go back to the agent for the next Thought
react_builder.add_edge("tools", "agent")

react_graph = react_builder.compile()


#### Run and Evaluate (Estimated Run Time 20min)

**Task:** Execute the ReAct agent pipeline on the dataset and collect SQL outputs.

In [90]:
from method_run import run_method
import re
def run_react_agent_with_config(item):
    question = item['question']
    schema = item['schema']
    user_prompt = f"Question: {question}\nSchema: {schema}"
    input_msg = HumanMessage(content=user_prompt)
    input_config = {"configurable": {"database_name": item['db_id']}}
    response = react_graph.invoke(MessagesState(messages=[input_msg]), config=input_config)

    for msg in response["messages"]:
        msg.pretty_print()
        
    # If last AI Message is a list of messages, we need to extract the last one
    last_msg = response["messages"][-1].content
    if isinstance(last_msg, list):
        last_msg = last_msg[-1]

    # First try to extract query from markdown SQL block
    match = re.search(r'```sql\n(.*?)```', last_msg, re.DOTALL)
    if match:
        query = match.group(1).strip()
    else:
        # If no markdown block found, try to extract just SQL query
        query = last_msg.strip()
        # Remove any ```sql or ``` if present without proper formatting
        query = re.sub(r'```sql|```', '', query).strip()

    return {**item, 'sql': query}

#Run agent on mode=nano, it's not needed to run on full dataset
run_method(run_react_agent_with_config, SLEEP_TIME=20, mode="nano")

  0%|          | 0/5 [00:00<?, ?it/s]

================================ Human Message =================================

Question: Find the percentage of atoms with single bond. (Evidence: single bond refers to bond_type = '-'; percentage = DIVIDE(SUM(bond_type = '-'), COUNT(bond_id)) as percentage)
Schema: atom (atom_id, molecule_id, element)
bond (bond_id, molecule_id, bond_type)
connected (atom_id, atom_id2, bond_id)
molecule (molecule_id, label)

================================== Ai Message ==================================
Tool Calls:
  get_foreign_keys (fb9ee533-1ba0-4aad-a36e-75bcb7cf04fa)
 Call ID: fb9ee533-1ba0-4aad-a36e-75bcb7cf04fa
  Args:
    table_name: connected
================================= Tool Message =================================
Name: get_foreign_keys

Error executing query: 
    SELECT
      tc.constraint_name,
      kcu.column_name AS source_column,
      ccu.table_name  AS target_table,
      ccu.column_name AS target_column
    FROM information_schema.table_constraints AS tc
    JOIN informa

 20%|██        | 1/5 [00:55<03:40, 55.18s/it]

================================ Human Message =================================

Question: What are the elements for bond id TR001_10_11? (Evidence: element = 'cl' means Chlorine; element = 'c' means Carbon; element = 'h' means Hydrogen; element = 'o' means Oxygen, element = 's' means Sulfur; element = 'n' means Nitrogen, element = 'p' means Phosphorus, element = 'na' means Sodium, element = 'br' means Bromine, element = 'f' means Fluorine; element = 'i' means Iodine; element = 'sn' means Tin; element = 'pb' means Lead; element = 'te' means Tellurium; element = 'ca' means Calcium)
Schema: atom (atom_id, molecule_id, element)
bond (bond_id, molecule_id, bond_type)
connected (atom_id, atom_id2, bond_id)
molecule (molecule_id, label)

================================== Ai Message ==================================
Tool Calls:
  execute (968b73e4-a9c9-4185-b69e-914771365ef3)
 Call ID: 968b73e4-a9c9-4185-b69e-914771365ef3
  Args:
    query: SELECT T1.element FROM atom AS T1 JOIN connected 

 40%|████      | 2/5 [01:39<02:26, 48.68s/it]

...

================================ Human Message =================================

Question: Find the name and date of events with expenses for pizza that were more than fifty dollars but less than a hundred dollars. (Evidence: name of event refers to event_name; date of event refers to event_date; expenses for pizza refers to expense_description = 'Pizza' where cost > 50 and cost < 100)
Schema: event (event_id, event_name, event_date, type, notes, location, status)
major (major_id, major_name, department, college)
zip_code (zip_code, type, city, county, state, short_state)
attendance (link_to_event, link_to_member)
budget (budget_id, category, spent, remaining, amount, event_status, link_to_event)
expense (expense_id, expense_description, expense_date, cost, approved, link_to_member, link_to_budget)
income (income_id, date_received, amount, source, notes, link_to_member)
member (member_id, first_name, last_name, email, position, t_shirt_size, phone, zip, link_to_major)

===============

 80%|████████  | 4/5 [03:06<00:45, 45.24s/it]

================================ Human Message =================================

Question: Based on the total cost for all event, what is the percentage of cost for Yearly Kickoff event? (Evidence: DIVIDE(SUM(cost where event_name = 'Yearly Kickoff'), SUM(cost)) * 100)
Schema: event (event_id, event_name, event_date, type, notes, location, status)
major (major_id, major_name, department, college)
zip_code (zip_code, type, city, county, state, short_state)
attendance (link_to_event, link_to_member)
budget (budget_id, category, spent, remaining, amount, event_status, link_to_event)
expense (expense_id, expense_description, expense_date, cost, approved, link_to_member, link_to_budget)
income (income_id, date_received, amount, source, notes, link_to_member)
member (member_id, first_name, last_name, email, position, t_shirt_size, phone, zip, link_to_major)

================================== Ai Message ==================================
Tool Calls:
  get_foreign_keys (47e7781a-e565-47e8-93

100%|██████████| 5/5 [04:16<00:00, 51.29s/it]


Starting to compare without knowledge for ex
Process finished successfully
start calculate
                     simple               moderate             challenging          total               
count                1                    1                    3                    5                   
======================================    ACCURACY    =====================================
accuracy             0.00                 0.00                 33.33                20.00               
Finished evaluation



https://chatgpt.com/share/68475f09-8b44-8010-b653-c60505c645db